# LSTM: Trajectory Regression + 3‑Klassen‑Wurfklassifikation
Dieses Notebook implementiert **genau die gewünschte Aufteilung**:
- **Modus A (noisy)**  (vollständige Sequenz → vollständige Sequenz)
- **Modus B (clean-trimmed / Extrapolation)**: `ExtrapSeq2SeqModel` (erste cleanen 30 Punkte konditionieren → **vollständig vorhergesagte** Trajektorie mit 50 Punkten)
- **Modus C (noisy-trimmed / Extrapolation)**: `ExtrapSeq2SeqModel` (erste noisy 30 Punkte konditionieren → **vollständig vorhergesagte** Trajektorie mit 50 Punkten)

Es erzeugt synthetische Ballistik‑Trajektorien, baut **Train/Validation/Test**‑Splits, trainiert die Modelle und evaluiert Regression & Klassifikation.


In [ ]:

# 0) Imports & Setup
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dataclasses import dataclass
from typing import Tuple, Optional, Dict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Reproducibility
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


## 1) DataGen: Ballistik + Labeling (0=zu kurz, 1=Treffer, 2=zu lang)

In [ ]:
# 1) Data generation (alternative generator: sequences end at hoop height on descending branch)
import numpy as np
import pandas as pd
import uuid


class DataGen:
    """
    Synthetic Free-Throw Generator (ballistic, no drag).
    """

    def __init__(self, hs=2.13, l=4.115, hKorb=3.048, Rr=0.2286, Rb=0.1219, g=9.807):
        self.params = {
            "l": float(l),
            "hKorb": float(hKorb),
            "Rr": float(Rr),
            "Rb": float(Rb),
            "g": float(g),
        }
        self.hs = float(hs)
        self.wurf_hoehe = 1.25 * self.hs  # release height

    def wurfgeschwindigkeit(self, theta):
        l = self.params["l"]
        h = self.params["hKorb"] - self.wurf_hoehe
        g = self.params["g"]
        return (l / np.cos(theta)) * np.sqrt(g / (2 * (l * np.tan(theta) - h)))

    def wurftrajektorien(self, theta, v0, t):
        g = self.params["g"]
        vx = v0 * np.cos(theta)
        vy = v0 * np.sin(theta)
        x = vx * t
        y = vy * t - 0.5 * g * t**2 + self.wurf_hoehe
        return x, y, vx, vy

    def _t_at_height_desc(self, theta, v0, y_target):
        g = self.params["g"]
        y0 = self.wurf_hoehe

        vx = v0 * np.cos(theta)
        vy = v0 * np.sin(theta)
        if vx <= 0:
            return None

        a = 0.5 * g
        b = -vy
        c = y_target - y0

        disc = b * b - 4 * a * c
        if disc <= 0:
            return None

        sqrt_disc = np.sqrt(disc)
        t1 = (-b - sqrt_disc) / (2 * a)
        t2 = (-b + sqrt_disc) / (2 * a)

        t_desc = max(t1, t2)
        if t_desc <= 0:
            return None
        return float(t_desc)

    def label_from_trajectory(self, theta, v0, margin=0.0):
        l = self.params["l"]
        h = self.params["hKorb"]
        Rr = self.params["Rr"]
        Rb = self.params["Rb"]

        vx = v0 * np.cos(theta)
        r_clear = max(0.0, (Rr - Rb) + margin)

        t_desc = self._t_at_height_desc(theta, v0, h)
        if t_desc is None or vx <= 0:
            return 0

        x_at_hoop_height = vx * t_desc

        if abs(x_at_hoop_height - l) <= r_clear:
            return 1
        elif x_at_hoop_height < l - r_clear:
            return 0
        else:
            return 2

    def generate_dataset_balanced(
        self,
        base_theta_deg=48.43,
        n_per_class=50,
        n_points=50,
        with_noise=False,
        noise_std=0.01,
        dtheta_range=(-0.20, 0.20),
        dv0_range=(-0.8, 0.8),
        margin=0.0,
        max_resample=5000,
        force_exact_endpoint=True,
    ):
        """
        Balanced dataset: exactly n_per_class throws for each label {0,1,2}.
        Returns a dataframe with columns compatible with the notebook pipeline.
        """
        base_theta = np.deg2rad(base_theta_deg)
        base_v0 = self.wurfgeschwindigkeit(base_theta)

        h = self.params["hKorb"]

        target_counts = {0: n_per_class, 1: n_per_class, 2: n_per_class}
        counts = {0: 0, 1: 0, 2: 0}

        rows = []
        wid = 0

        # generate in rounds: 0,1,2,0,1,2,... until all filled
        desired_labels = [0, 1, 2]

        while any(counts[k] < target_counts[k] for k in counts):
            for desired in desired_labels:
                if counts[desired] >= target_counts[desired]:
                    continue

                theta = v0 = T = None

                for _ in range(max_resample):
                    dth = np.random.uniform(*dtheta_range)
                    dv0 = np.random.uniform(*dv0_range)
                    theta_try = base_theta + dth
                    v0_try = base_v0 + dv0

                    t_desc = self._t_at_height_desc(theta_try, v0_try, h)
                    if t_desc is None:
                        continue

                    label_try = self.label_from_trajectory(theta_try, v0_try, margin=margin)
                    if label_try == desired:
                        theta, v0, T = theta_try, v0_try, t_desc
                        label = label_try
                        break

                if T is None:
                    raise RuntimeError(
                        f"Could not sample enough examples for class {desired} "
                        f"within max_resample={max_resample}. "
                        f"Try widening dtheta_range/dv0_range or increasing max_resample."
                    )

                t_vals = np.linspace(0.0, T, n_points, dtype=np.float32)
                x, y, vx, vy = self.wurftrajektorien(theta, v0, t_vals)

                x_clean = x.copy()
                y_clean = y.copy()

                if force_exact_endpoint:
                    vx0 = v0 * np.cos(theta)
                    x_clean[-1] = float(vx0 * T)
                    y_clean[-1] = float(h)

                if with_noise:
                    x_noisy = x_clean + np.random.normal(0.0, noise_std, size=x_clean.shape)
                    y_noisy = y_clean + np.random.normal(0.0, noise_std, size=y_clean.shape)
                    if force_exact_endpoint:
                        x_noisy[-1] = x_clean[-1]
                        y_noisy[-1] = y_clean[-1]
                else:
                    x_noisy = x_clean.copy()
                    y_noisy = y_clean.copy()

                x_sel = x_noisy if with_noise else x_clean
                y_sel = y_noisy if with_noise else y_clean

                for ti, xc, yc, xn, yn, xi, yi in zip(t_vals, x_clean, y_clean, x_noisy, y_noisy, x_sel, y_sel):
                    rows.append(
                        {
                            "wurf_id": wid,
                            "t": float(ti),
                            "x_clean": float(xc),
                            "y_clean": float(yc),
                            "x_noisy": float(xn),
                            "y_noisy": float(yn),
                            "x": float(xi),
                            "y": float(yi),
                            "T": float(T),
                            "label": int(label),
                            "theta": float(theta),
                            "v0": float(v0),
                        }
                    )

                counts[desired] += 1
                wid += 1

        return pd.DataFrame(rows)


def add_point_index_and_tnorm(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["point_index"] = df.groupby("wurf_id").cumcount()
    # Normiere t pro Wurf auf [0,1]
    tmax = df.groupby("wurf_id")["t"].transform("max")
    df["t_norm"] = df["t"] / tmax
    return df


## 2) Train/Val/Test: getrennt generieren (je Split balanced)

In [ ]:
# Generate balanced datasets (each trajectory ends at hoop height on the descending branch)

dg = DataGen()

# Anzahl Würfe pro Klasse pro Split
N_TRAIN_PER_CLASS = 400
N_VAL_PER_CLASS   = 120
N_TEST_PER_CLASS  = 120

# Reproducible splits (this generator uses NumPy's global RNG)
np.random.seed(1)
df_train = dg.generate_dataset_balanced(
    base_theta_deg=50.0,
    n_per_class=N_TRAIN_PER_CLASS,
    n_points=50,
    with_noise=True,
    noise_std=0.2,
    dtheta_range=(-0.21, 0.21),   # ~±12°
    dv0_range=(-1.2, 1.2),
    margin=0.0,
    max_resample=20000,
    force_exact_endpoint=True,
)
df_train = add_point_index_and_tnorm(df_train)

np.random.seed(2)
df_val = dg.generate_dataset_balanced(
    base_theta_deg=50.0,
    n_per_class=N_VAL_PER_CLASS,
    n_points=50,
    with_noise=True,
    noise_std=0.2,
    dtheta_range=(-0.21, 0.21),
    dv0_range=(-1.2, 1.2),
    margin=0.0,
    max_resample=20000,
    force_exact_endpoint=True,
)
df_val = add_point_index_and_tnorm(df_val)

np.random.seed(3)
df_test = dg.generate_dataset_balanced(
    base_theta_deg=50.0,
    n_per_class=N_TEST_PER_CLASS,
    n_points=50,
    with_noise=True,
    noise_std=0.2,
    dtheta_range=(-0.21, 0.21),
    dv0_range=(-1.2, 1.2),
    margin=0.0,
    max_resample=20000,
    force_exact_endpoint=True,
)
df_test = add_point_index_and_tnorm(df_test)

df_train.shape, df_val.shape, df_test.shape


In [ ]:

# Check Label-Verteilung (pro Wurf)
def wurf_label_counts(df):
    w = df.drop_duplicates("wurf_id")[["wurf_id","label"]]
    return w["label"].value_counts().sort_index()

print("Train:", wurf_label_counts(df_train).to_dict())
print("Val:  ", wurf_label_counts(df_val).to_dict())
print("Test: ", wurf_label_counts(df_test).to_dict())


## 3) Scaling (fit auf Train)

In [ ]:

class XYScaler:
    def __init__(self):
        self.mean_ = None
        self.std_ = None

    def fit(self, xy: np.ndarray):
        self.mean_ = xy.mean(axis=0, keepdims=True)
        self.std_ = xy.std(axis=0, keepdims=True) + 1e-8
        return self

    def transform(self, xy: np.ndarray) -> np.ndarray:
        return (xy - self.mean_) / self.std_

    def inverse_transform(self, xy_scaled: np.ndarray) -> np.ndarray:
        return xy_scaled * self.std_ + self.mean_

# Fit auf clean targets aus Train
train_xy = df_train[["x_clean","y_clean"]].to_numpy(np.float32)
scaler = XYScaler().fit(train_xy)

# Quick sanity: Standardisierung
xy_s = scaler.transform(train_xy)
xy_s.mean(axis=0), xy_s.std(axis=0)


## 4) Dataset

In [ ]:

def df_to_wurf_arrays(df: pd.DataFrame):
    # group each wurf into arrays ordered by point_index
    groups = []
    for wid, g in df.groupby("wurf_id"):
        g = g.sort_values("point_index")
        groups.append(g)
    return groups


class ExtrapolationDataset(Dataset):
    """Für ExtrapSeq2SeqModel (Option 2): observed Tin → **vollständig vorhergesagte** Trajektorie (T).

    - Encoder sieht die ersten Tin Punkte (x,y,t_norm).
    - Decoder erzeugt **alle** T Punkte autoregressiv.
    - Targets (y_all) enthalten die komplette Trajektorie (T,2).
    """
    def __init__(
        self,
        df: pd.DataFrame,
        scaler: XYScaler,
        tin: int = 30,
        input_source: str = "noisy",   # "clean" | "noisy"
        target_source: str = "clean"   # "clean" | "noisy"
    ):
        assert input_source in {"clean","noisy"}
        assert target_source in {"clean","noisy"}
        self.scaler = scaler
        self.tin = tin
        self.input_source = input_source
        self.target_source = target_source

        self.groups = df_to_wurf_arrays(df)

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, idx):
        g = self.groups[idx]
        label = int(g["label"].iloc[0])

        T = len(g)
        assert self.tin <= T

        t_norm = g["t_norm"].to_numpy(np.float32)[:, None]  # (T,1)

        xy_clean = g[["x_clean","y_clean"]].to_numpy(np.float32)
        xy_noisy = g[["x_noisy","y_noisy"]].to_numpy(np.float32)

        xy_in  = xy_noisy if self.input_source=="noisy" else xy_clean
        xy_tgt = xy_clean if self.target_source=="clean" else xy_noisy

        xy_in_s  = self.scaler.transform(xy_in)    # (T,2)
        xy_tgt_s = self.scaler.transform(xy_tgt)   # (T,2)

        # Encoder input: (Tin,3) = [x,y,t_norm]
        x_in = np.concatenate([xy_in_s[:self.tin], t_norm[:self.tin]], axis=1).astype(np.float32)

        # Decoder times for *all* steps: (T,1)
        t_all = t_norm.astype(np.float32)

        # Full targets: (T,2)
        y_all = xy_tgt_s.astype(np.float32)

        return (
            torch.from_numpy(x_in),                 # (Tin,3)
            torch.from_numpy(t_all),                # (T,1)
            torch.from_numpy(y_all),                # (T,2)
            torch.tensor(label, dtype=torch.long)
        )


def collate_extrap(batch):
    # Pads are not needed: Tin and T are fixed by construction.
    x_in, t_all, y_all, labels = zip(*batch)
    return (
        torch.stack(x_in, dim=0),     # (B,Tin,3)
        torch.stack(t_all, dim=0),    # (B,T,1)
        torch.stack(y_all, dim=0),    # (B,T,2)
        torch.stack(labels, dim=0),   # (B,)
    )

## 5) Modelle

In [ ]:
class ExtrapSeq2SeqModel(nn.Module):
    """
    Option 2 (vollständig predicted):
    - Encoder sieht beobachtete Punkte (x,y,t_norm) der Länge Tin.
    - Decoder läuft autoregressiv über **alle** T Zeitschritte und erzeugt (x,y).
      Decoder-Input pro Schritt: [prev_x, prev_y, t_k] (3 Werte).

    Teacher forcing (Training): prev_xy = y_all[:,k] (Ground Truth des aktuellen Schritts)
    Autoregressiv (Eval): prev_xy = xy_k (eigene Vorhersage)
    """

    def __init__(self, input_dim=3, hidden_dim=128, num_layers=2, n_classes=3, dropout=0.1):
        super().__init__()
        self.enc = nn.LSTM(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=(dropout if num_layers > 1 else 0.0),
        )
        self.dec_cell = nn.LSTMCell(input_size=3, hidden_size=hidden_dim)
        self.out_head = nn.Linear(hidden_dim, 2)
        self.cls_head = nn.Linear(hidden_dim, n_classes)

        # Learnable start token for the decoder (scaled xy)
        self.start_xy = nn.Parameter(torch.zeros(2))

    def forward(self, x_in, t_all, y_all=None, teacher_forcing=True):
        """
        x_in:  (B,Tin,3) = [x,y,t_norm] (observed)
        t_all: (B,T,1)   = all normalized times
        y_all: (B,T,2)   = full ground truth (optional, for teacher forcing)

        returns:
          y_hat:  (B,T,2)  full predicted trajectory (scaled)
          logits: (B,3)
        """
        out, (hN, cN) = self.enc(x_in)      # out: (B,Tin,H)
        h = hN[-1]                           # (B,H)
        c = cN[-1]

        logits = self.cls_head(h)           # (B,3)

        B = x_in.size(0)
        T = t_all.size(1)

        preds = []
        prev_xy = self.start_xy.unsqueeze(0).expand(B, -1)  # (B,2)

        for k in range(T):
            dec_in = torch.cat([prev_xy, t_all[:, k, :]], dim=1)  # (B,3)
            h, c = self.dec_cell(dec_in, (h, c))
            xy_k = self.out_head(h)                               # (B,2)
            preds.append(xy_k.unsqueeze(1))

            if teacher_forcing and (y_all is not None):
                prev_xy = y_all[:, k, :]
            else:
                prev_xy = xy_k

        y_hat = torch.cat(preds, dim=1)  # (B,T,2)
        return y_hat, logits

## 6) Training & Evaluation Helpers

In [ ]:

def mse_loss(pred, target):
    return F.mse_loss(pred, target)

def train_extrap(
    model: ExtrapSeq2SeqModel,
    train_loader: DataLoader,
    val_loader: DataLoader,
    tin: int = 30,
    epochs: int = 25,
    lr: float = 2e-3,
    cls_weight: float = 0.2,
    teacher_forcing: bool = True,
    verbose: bool = True,
):
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    best = {"loss": 1e18, "state": None}
    hist = {"tr_loss": [], "va_loss": [], "tr_acc": [], "va_acc": [], "tr_mse_future": [], "va_mse_future": []}

    for ep in range(1, epochs + 1):
        # Train
        model.train()
        tr_losses, tr_y, tr_p, tr_mse_f = [], [], [], []
        for x_in, t_all, y_all, c in train_loader:
            x_in = x_in.to(device)
            t_all = t_all.to(device)
            y_all = y_all.to(device)
            c = c.to(device)

            yhat, logits = model(x_in, t_all, y_all=y_all, teacher_forcing=teacher_forcing)

            T = y_all.size(1)

            if tin >= T:
                future_mse = 0.0
            else:
                future_mse = F.mse_loss(yhat[:, tin:, :], y_all[:, tin:, :]).item()

            loss_reg = F.mse_loss(yhat, y_all)
            loss_cls = F.cross_entropy(logits, c)
            loss = loss_reg + cls_weight * loss_cls

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            
            tr_losses.append(loss.item())
            # Future-only mse (reporting)
            tr_mse_f.append(F.mse_loss(yhat[:, tin:, :], y_all[:, tin:, :]).item())

            tr_y.extend(c.detach().cpu().tolist())
            tr_p.extend(torch.argmax(logits, dim=1).detach().cpu().tolist())

        tr_acc = accuracy_score(tr_y, tr_p)

        # Val (autoregressiv)
        model.eval()
        va_losses, va_y, va_p, va_mse_f = [], [], [], []
        with torch.no_grad():
            for x_in, t_all, y_all, c in val_loader:
                x_in = x_in.to(device)
                t_all = t_all.to(device)
                y_all = y_all.to(device)
                c = c.to(device)

                yhat, logits = model(x_in, t_all, y_all=None, teacher_forcing=False)

                T = y_all.size(1)

                if tin >= T:
                    future_mse = 0.0
                else:
                    future_mse = F.mse_loss(yhat[:, tin:, :], y_all[:, tin:, :]).item()

                va_mse_f.append(future_mse)


                loss_reg = F.mse_loss(yhat, y_all)
                loss_cls = F.cross_entropy(logits, c)
                loss = loss_reg + cls_weight * loss_cls

                va_losses.append(loss.item())
                va_mse_f.append(F.mse_loss(yhat[:, tin:, :], y_all[:, tin:, :]).item())

                va_y.extend(c.cpu().tolist())
                va_p.extend(torch.argmax(logits, dim=1).cpu().tolist())

        va_acc = accuracy_score(va_y, va_p)

        tr_loss = float(np.mean(tr_losses))
        va_loss = float(np.mean(va_losses))

        hist["tr_loss"].append(tr_loss)
        hist["va_loss"].append(va_loss)
        hist["tr_acc"].append(tr_acc)
        hist["va_acc"].append(va_acc)
        hist["tr_mse_future"].append(float(np.mean(tr_mse_f)))
        hist["va_mse_future"].append(float(np.mean(va_mse_f)))

        if va_loss < best["loss"]:
            best["loss"] = va_loss
            best["state"] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if verbose:
            print(
                f"[C] ep={ep:02d} tr_loss={tr_loss:.4f} va_loss={va_loss:.4f} "
                f"| tr_acc={tr_acc:.3f} va_acc={va_acc:.3f} "
                f"| va_future_mse={hist['va_mse_future'][-1]:.5f}"
            )

    if best["state"] is not None:
        model.load_state_dict(best["state"])
    return hist


def eval_extrap(model: ExtrapSeq2SeqModel, loader: DataLoader, tin: int = 30, split_name="test"):
    model.eval()
    all_y, all_p = [], []
    mse_full, mse_future = [], []
    with torch.no_grad():
        for x_in, t_all, y_all, c in loader:
            x_in = x_in.to(device)
            t_all = t_all.to(device)
            y_all = y_all.to(device)
            c = c.to(device)

            yhat, logits = model(x_in, t_all, y_all=None, teacher_forcing=False)

            mse_full.append(F.mse_loss(yhat, y_all).item())
            mse_future.append(F.mse_loss(yhat[:, tin:, :], y_all[:, tin:, :]).item())

            all_y.extend(c.cpu().tolist())
            all_p.extend(torch.argmax(logits, dim=1).cpu().tolist())

    acc = accuracy_score(all_y, all_p)
    cm = confusion_matrix(all_y, all_p, labels=[0,1,2])

    print(f"[{split_name}] full_mse_scaled={float(np.mean(mse_full)):.5f} | future_mse_scaled={float(np.mean(mse_future)):.5f} | acc={acc:.3f}")
    print(classification_report(all_y, all_p, digits=3))

    return cm


def plot_cm(cm, title="Confusion Matrix"):
    fig = plt.figure(figsize=(4,4))
    plt.imshow(cm)
    plt.title(title)
    plt.xlabel("pred")
    plt.ylabel("true")
    plt.xticks([0,1,2])
    plt.yticks([0,1,2])
    for i in range(3):
        for j in range(3):
            plt.text(j, i, cm[i,j], ha="center", va="center")
    plt.show()


def plot_history(hist: dict, title: str):
    """
    Erwartet typischerweise:
      hist = {
        "train_loss": [...], "val_loss": [...],
        "train_acc": [...],  "val_acc": [...]
      }
    Optional werden zusätzliche bekannte Keys automatisch geplottet.
    """

    def _plot(keys, subtitle):
        present = [k for k in keys if k in hist and len(hist[k]) > 0]
        if not present:
            return
        plt.figure(figsize=(10, 4))
        for k in present:
            plt.plot(hist[k], label=k)
        plt.title(subtitle)
        plt.xlabel("epoch")
        plt.legend()
        plt.grid(True, alpha=0.2)
        plt.show()

    _plot(["train_loss", "val_loss"], title)
    _plot(["train_acc", "val_acc"], title + " (accuracy)")

    # Optional: falls du später mehr protokollierst
    _plot(["train_mse", "val_mse"], title + " (mse)")
    _plot(["train_cls_loss", "val_cls_loss"], title + " (cls loss)")
    _plot(["train_reg_loss", "val_reg_loss"], title + " (reg loss)")



## 7) Modus A (noisy) mit  ExtrapSeq2SeqModel
Input & Target sind noisy.
Encoder bekommt **Tin=50** Punkte

In [ ]:
batch_size = 64

tin = 50
dsA_train = ExtrapolationDataset(df_train, scaler, tin=tin, input_source="noisy", target_source="noisy")
dsA_val   = ExtrapolationDataset(df_val, scaler, tin=tin, input_source="noisy", target_source="noisy")
dsA_test  = ExtrapolationDataset(df_test, scaler, tin=tin, input_source="noisy", target_source="noisy")

dlA_train = DataLoader(dsA_train, batch_size=batch_size, shuffle=True, drop_last=False, collate_fn=collate_extrap)
dlA_val   = DataLoader(dsA_val,   batch_size=batch_size, shuffle=False, collate_fn=collate_extrap)
dlA_test  = DataLoader(dsA_test,  batch_size=batch_size, shuffle=False, collate_fn=collate_extrap)

modelA = ExtrapSeq2SeqModel(hidden_dim=128, num_layers=2, dropout=0.1).to(device)

histA = train_extrap(
    modelA, dlA_train, dlA_val,
    tin=tin,
    epochs=25, lr=2e-3, cls_weight=0.2,
    teacher_forcing=True
)

plot_history(histA, "Mode A (extrapolation) - ExtrapSeq2SeqModel")

cmA = eval_extrap(modelA, dlA_test, tin=tin, split_name="test A")
plot_cm(cmA, "Mode A (extrapolation) - Test Confusion Matrix")

## 8) Modus B (clean-trimmed/extrapolation) mit ExtrapSeq2SeqModel
Encoder bekommt **Tin=30** Punkte, Decoder erzeugt eine **vollständige, komplett vorhergesagte** Trajektorie (T=50 Punkte).

In [ ]:

tin = 30
dsB_train = ExtrapolationDataset(df_train, scaler, tin=tin, input_source="clean", target_source="clean")
dsB_val   = ExtrapolationDataset(df_val, scaler, tin=tin, input_source="clean", target_source="clean")
dsB_test  = ExtrapolationDataset(df_test, scaler, tin=tin, input_source="clean", target_source="clean")

dlB_train = DataLoader(dsB_train, batch_size=batch_size, shuffle=True, drop_last=False, collate_fn=collate_extrap)
dlB_val   = DataLoader(dsB_val,   batch_size=batch_size, shuffle=False, collate_fn=collate_extrap)
dlB_test  = DataLoader(dsB_test,  batch_size=batch_size, shuffle=False, collate_fn=collate_extrap)

modelB = ExtrapSeq2SeqModel(hidden_dim=128, num_layers=2, dropout=0.1).to(device)

histB = train_extrap(
    modelB, dlB_train, dlB_val,
    tin=tin,
    epochs=25, lr=2e-3, cls_weight=0.2,
    teacher_forcing=True
)

plot_history(histB, "Mode B (extrapolation) - ExtrapSeq2SeqModel")

cmB = eval_extrap(modelB, dlB_test, tin=tin, split_name="test B")
plot_cm(cmB, "Mode B (extrapolation) - Test Confusion Matrix")

## 9) Modus C (noisy-trimmed/extrapolation) mit ExtrapSeq2SeqModel
Input & Target sind noisy.
Encoder bekommt **Tin=30** noisy Punkte, Decoder erzeugt eine **vollständige, komplett vorhergesagte** Trajektorie (T=50 Punkte).

In [ ]:

tin = 30
dsC_train = ExtrapolationDataset(df_train, scaler, tin=tin, input_source="noisy", target_source="noisy")
dsC_val   = ExtrapolationDataset(df_val, scaler, tin=tin, input_source="noisy", target_source="noisy")
dsC_test  = ExtrapolationDataset(df_test, scaler, tin=tin, input_source="noisy", target_source="noisy")

dlC_train = DataLoader(dsC_train, batch_size=batch_size, shuffle=True, drop_last=False, collate_fn=collate_extrap)
dlC_val   = DataLoader(dsC_val,   batch_size=batch_size, shuffle=False, collate_fn=collate_extrap)
dlC_test  = DataLoader(dsC_test,  batch_size=batch_size, shuffle=False, collate_fn=collate_extrap)

modelC = ExtrapSeq2SeqModel(hidden_dim=128, num_layers=2, dropout=0.1).to(device)

histC = train_extrap(
    modelC, dlC_train, dlC_val,
    tin=tin,
    epochs=25, lr=2e-3, cls_weight=0.2,
    teacher_forcing=True
)

plot_history(histC, "Mode C (extrapolation) - ExtrapSeq2SeqModel")

cmC = eval_extrap(modelC, dlC_test, tin=tin, split_name="test C")
plot_cm(cmC, "Mode C (extrapolation) - Test Confusion Matrix")

## 10) Qualitative Plots: Trajektorien (Ground Truth vs Prediction)
Wir plotten pro Modus ein Beispiel aus Test.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch


def _nice_limits_xy(*curves_xy, pad_frac=0.06, pad_abs=0.03):
    """
    curves_xy: Arrays (N,2) in meters.
    Returns xlim, ylim with a small padding like in the example plot.
    """
    pts = [c for c in curves_xy if c is not None and len(c) > 0]
    if not pts:
        return None, None
    allp = np.vstack(pts)
    xmin, ymin = np.min(allp, axis=0)
    xmax, ymax = np.max(allp, axis=0)

    dx = max(1e-6, xmax - xmin)
    dy = max(1e-6, ymax - ymin)

    xpad = max(pad_abs, pad_frac * dx)
    ypad = max(pad_abs, pad_frac * dy)

    return (xmin - xpad, xmax + xpad), (ymin - ypad, ymax + ypad)


def _draw_rim(ax, dg):
    # rim segment at hoop height (like in your screenshot)
    h = dg.params["hKorb"]
    l = dg.params["l"]
    rr = dg.params["Rr"]
    ax.hlines(h, l - rr, l + rr, linewidth=3, label="rim (10ft)")


def plot_example_A(
    model,
    ds,
    scaler,
    dg,
    device,
    n=3,
    title="Complete Noisy Observation",
):
    model.eval()
    idxs = np.random.choice(len(ds), size=n, replace=False)

    for idx in idxs:
        x, y, c = ds[idx]  # x: (T,F), y: (T,2), c: ()
        x_np = x.numpy()
        y_np = y.numpy()

        with torch.no_grad():
            yhat, logits = model(x.unsqueeze(0).to(device))
        yhat_np = yhat.squeeze(0).cpu().numpy()
        pred_c = int(torch.argmax(logits, dim=1).cpu().item())

        y_m = scaler.inverse_transform(y_np)
        yhat_m = scaler.inverse_transform(yhat_np)

        if x_np.shape[1] >= 4:
            known_mask = x_np[:, 3] > 0.5
        else:
            known_mask = np.ones(x_np.shape[0], dtype=bool)

        obs_xy_scaled = x_np[known_mask, :2]
        obs_xy_m = (
            scaler.inverse_transform(obs_xy_scaled) if len(obs_xy_scaled) else None
        )

        # ---- plot ----
        fig, ax = plt.subplots(figsize=(6, 4.7))
        ax.grid(True, alpha=0.2)

        if obs_xy_m is not None and len(obs_xy_m) > 0:
            ax.plot(
                obs_xy_m[:, 0],
                obs_xy_m[:, 1],
                "o",
                markersize=4,
                label="observed input points",
            )

        ax.plot(y_m[:, 0], y_m[:, 1], label="ground truth (target)")
        ax.plot(yhat_m[:, 0], yhat_m[:, 1], linestyle="--", label="prediction")

        _draw_rim(ax, dg)

        ax.set_xlabel("x [m]")
        ax.set_ylabel("y [m]")

        # dynamic limits (like example)
        xlim, ylim = _nice_limits_xy(y_m, yhat_m, obs_xy_m)
        if xlim:
            ax.set_xlim(*xlim)
        if ylim:
            ax.set_ylim(*ylim)

        ax.set_title(f"{title} (GT={int(c)}, Pred={pred_c})")
        ax.legend(loc="lower center", fontsize=9, framealpha=0.9)

        plt.tight_layout()
        plt.show()


def plot_example_B(
    model,
    ds,
    scaler,
    dg,
    device,
    tin: int = 30,
    n=3,
    title="Clean/Trimmed Observation",
):
    model.eval()
    idxs = np.random.choice(len(ds), size=n, replace=False)

    for idx in idxs:
        x_in, t_all, y_all, c = ds[idx]  # x_in: (Tin,3), t_all: (T,1), y_all: (T,2)
        with torch.no_grad():
            yhat, logits = model(
                x_in.unsqueeze(0).to(device),
                t_all.unsqueeze(0).to(device),
                y_all=None,
                teacher_forcing=False,
            )

        yhat_np = yhat.squeeze(0).cpu().numpy()
        y_np = y_all.numpy()
        pred_c = int(torch.argmax(logits, dim=1).cpu().item())

        # meters
        y_m = scaler.inverse_transform(y_np)
        yhat_m = scaler.inverse_transform(yhat_np)

        # observed points (context)
        obs_xy_scaled = x_in[:, :2].numpy()
        obs_xy_m = scaler.inverse_transform(obs_xy_scaled)

        x_end = obs_xy_m[-1, 0] if len(obs_xy_m) > 0 else None

        fig, ax = plt.subplots(figsize=(6, 4.7))
        ax.grid(True, alpha=0.2)

        ax.plot(
            obs_xy_m[:, 0],
            obs_xy_m[:, 1],
            "o",
            markersize=4,
            label="observed input points",
        )
        if x_end is not None:
            ax.axvline(x_end, linestyle=":", label="end of observations")

        ax.plot(y_m[:, 0], y_m[:, 1], label="ground truth (target)")
        ax.plot(yhat_m[:, 0], yhat_m[:, 1], linestyle="--", label="prediction")

        _draw_rim(ax, dg)

        ax.set_xlabel("x [m]")
        ax.set_ylabel("y [m]")

        xlim, ylim = _nice_limits_xy(y_m, yhat_m, obs_xy_m)
        if xlim:
            ax.set_xlim(*xlim)
        if ylim:
            ax.set_ylim(*ylim)

        ax.set_title(f"{title} (GT={int(c)}, Pred={pred_c})")
        ax.legend(loc="lower center", fontsize=9, framealpha=0.9)

        plt.tight_layout()
        plt.show()


plot_example_A(
    modelA,
    dsA_test,
    scaler,
    dg,
    device,
    n=3,
    title="Mode A: Complete Noisy Observation",
)

plot_example_B(
    modelB,
    dsB_test,
    scaler,
    dg,
    device,
    tin=tin,
    n=3,
    title="Mode B: Clean/Trimmed Observation",
)